# Regional Inertia Estimation

The Inertia Estimator aggregates generator power, tie-line flow and frequency for one WECC region
(DEMO2's fixed region definitions), then fits a Kalman-filter regression of power deviation on
rate of change of frequency. It publishes:

- `regional_signal`: the regional frequency deviation (pu) and power deviation (MW) for every sample;
- `inertia`: one estimate (mean and median kinetic energy, MVA·s) each time a window of
  `window_sec` seconds closes.

The templates replay DEMO2 PSS/E snapshots of 16 s each: either one hour (CA1, day 6 hour 1) or all
24 hours of day 6, one snapshot per hour of the day, so the run produces one estimate per snapshot.
Estimates are compared with the DEMO2 ground truth below. The estimator's known limitations are
listed in `KNOWN_ISSUES.md` in the `oedisi-inertia` repository.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.feather as pf

# DATA_DIR will be set automatically when copied to a run
DATA_DIR = r""

RUN_DIR = Path(DATA_DIR) if DATA_DIR else Path.cwd()
OUTPUT_DIR = RUN_DIR / "notebook_outputs"

# Payload times count from this instant (ParaEMT's default start; the Player data uses it too).
EPOCH = pd.Timestamp("2026-01-01 00:00:00")

BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e"

print(f"Run directory: {RUN_DIR}")

## Load the recorded outputs

The run's `wiring.json` says which Recorder is attached to which estimator output, and which
Players feed the estimator, so component names don't matter. Times are converted to seconds.

In [ ]:
def find_run_parts(run_dir: Path):
    """Return the estimator component, {output port: feather path}, and the Players feeding it."""
    wiring = json.loads((run_dir / "wiring.json").read_text(encoding="utf-8"))
    components = {c["name"]: c for c in wiring["components"]}
    estimators = [
        c for c in wiring["components"]
        if {"region", "window_sec"} <= set(c.get("parameters", {}))
    ]
    if len(estimators) != 1:
        raise ValueError(f"Expected one Inertia Estimator in wiring.json; found {len(estimators)}.")
    estimator = estimators[0]

    recordings, players = {}, []
    for link in wiring["links"]:
        if link["source"] == estimator["name"] and components[link["target"]]["type"] == "Recorder":
            recorder = components[link["target"]]
            recordings[link["source_port"]] = (
                run_dir / "build" / recorder["name"] / recorder["parameters"]["feather_filename"]
            )
        elif link["target"] == estimator["name"] and "filename" in components[link["source"]]["parameters"]:
            players.append(components[link["source"]])
    return estimator, recordings, players


def load_output(recordings: dict[str, Path], port: str) -> pd.DataFrame:
    if port not in recordings:
        raise KeyError(f"No Recorder is attached to the estimator's '{port}' output. Add one and run again.")
    path = recordings[port]
    if not path.is_file():
        raise FileNotFoundError(f"{port}: {path} does not exist.")
    frame = pf.read_feather(path)
    frame["seconds"] = (pd.to_datetime(frame["time"]) - EPOCH).dt.total_seconds()
    return frame


estimator, recordings, players = find_run_parts(RUN_DIR)
REGION = estimator["parameters"]["region"]
WINDOW_SEC = float(estimator["parameters"]["window_sec"])

regional = load_output(recordings, "regional_signal")
inertia = load_output(recordings, "inertia").reset_index(drop=True)

if regional.empty or not regional["seconds"].is_monotonic_increasing:
    raise ValueError("regional_signal is empty or its time is not monotonic.")
# One regional sample per Player row; the last may be missing if the run stops one step early.
expected_rows = min(int(p["parameters"]["number_of_timesteps"]) for p in players) if players else None
if expected_rows is not None and len(regional) < expected_rows - 1:
    raise ValueError(f"regional_signal has {len(regional)} rows; the Players sent {expected_rows}.")
if inertia.empty:
    raise ValueError(
        f"No inertia estimate was published: no {WINDOW_SEC:g} s window closed "
        "(partial windows are discarded)."
    )

# Each regional sample belongs to the first window ending at or after it.
window_index = np.searchsorted(inertia["seconds"].to_numpy(), regional["seconds"].to_numpy(), side="left")
regional["window"] = window_index
regional = regional[regional["window"] < len(inertia)]

print(f"Region: {REGION}; window: {WINDOW_SEC:g} s")
print(f"Regional samples: {len(regional):,}; inertia estimates: {len(inertia)}")

## Ground truth

DEMO2's hourly regional inertia (the H·MBASE sum over the units running in that hour) is kept
beside the replayed data as `ground_truth.json` and read here only; the estimator never sees it.
Each estimate is matched to the snapshot whose start precedes its window end. If the input isn't a
bundled scenario, only the estimates are shown.

In [ ]:
def load_ground_truth(players: list[dict], region: str) -> list[dict]:
    """Ground-truth snapshots as [{hour_num, start_sec, true_energy_mva_s}], or [] if none."""
    for player in players:
        path = Path(player["parameters"]["filename"]).expanduser().parent / "ground_truth.json"
        if not path.is_file():
            continue
        truth = json.loads(path.read_text(encoding="utf-8"))
        if truth.get("region") != region:
            continue
        if "hours" in truth:
            return truth["hours"]
        hour = int(truth.get("hour_case", "hour_1").rsplit("_", 1)[-1])
        return [{"hour_num": hour, "start_sec": 0.0, "true_energy_mva_s": truth["true_energy_mva_s"]}]
    return []


snapshots = sorted(load_ground_truth(players, REGION), key=lambda s: s["start_sec"])

rows = []
for i, estimate in inertia.iterrows():
    match = [s for s in snapshots if s["start_sec"] - 1.0 <= estimate["seconds"]]
    snapshot = match[-1] if match else None
    rows.append({
        "window": i + 1,
        "hour": snapshot["hour_num"] if snapshot else np.nan,
        "window end": estimate["time"],
        "mean (MVA·s)": float(estimate["mean_energy_mva_s"]),
        "median (MVA·s)": float(estimate["median_energy_mva_s"]),
        "ground truth (MVA·s)": snapshot["true_energy_mva_s"] if snapshot else np.nan,
    })
results = pd.DataFrame(rows)
HAS_TRUTH = results["ground truth (MVA·s)"].notna().all()
if HAS_TRUTH:
    for kind in ("mean", "median"):
        error = results[f"{kind} (MVA·s)"] - results["ground truth (MVA·s)"]
        results[f"{kind} error (%)"] = 100 * error / results["ground truth (MVA·s)"]
else:
    print(f"No ground_truth.json for region {REGION} next to the Player data.")

results.style.format(precision=1, thousands=",").hide(axis="index")

## Estimates vs. ground truth

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if len(results) == 1:
    last = results.iloc[0]
    labels = ["Mean estimate", "Median estimate"]
    values = [last["mean (MVA·s)"], last["median (MVA·s)"]]
    truth = last["ground truth (MVA·s)"] if HAS_TRUTH else None

    fig, ax = plt.subplots(figsize=(9, 2.8), constrained_layout=True)
    bars = ax.barh(labels, values, color=BLUE, height=0.5)
    ax.invert_yaxis()
    for bar, value in zip(bars, values):
        note = f"{value:,.0f}"
        if truth is not None:
            note += f"  ({100 * (value - truth) / truth:+.1f}%)"
        ax.text(value * 0.98, bar.get_y() + bar.get_height() / 2, note,
                ha="right", va="center", color="white", fontsize=10)
    if truth is not None:
        ax.axvline(truth, color=INK, linestyle="--", linewidth=1.5)
        ax.text(truth * 1.01, 0.5, f"Ground truth\n{truth:,.0f}", color=INK, va="center", ha="left", fontsize=10)
    ax.set_xlim(0, max(values + ([truth] if truth else [])) * 1.18)
    ax.xaxis.set_major_formatter(lambda x, _: f"{x:,.0f}")
    ax.set_xlabel("Kinetic energy (MVA·s)")
    ax.set_title(f"{REGION} inertia estimate", loc="left", color=INK)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, axis="x", alpha=0.25)
else:
    x = results["hour"] if results["hour"].notna().all() else results["window"]
    fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
    ax.plot(x, results["mean (MVA·s)"], "-o", color=BLUE, linewidth=2, markersize=6, label="Mean KF estimate")
    ax.plot(x, results["median (MVA·s)"], "-s", color=ORANGE, linewidth=2, markersize=6, label="Median KF estimate")
    if HAS_TRUTH:
        ax.plot(x, results["ground truth (MVA·s)"], "--^", color=INK, linewidth=1.5, markersize=6, label="Ground truth")
    ax.set_xticks(list(x))
    ax.set_xlabel("Hour" if results["hour"].notna().all() else "Window")
    ax.set_ylabel("Kinetic energy (MVA·s)")
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
    ax.set_title(f"{REGION} hourly Kalman-filtered inertia vs. ground truth", loc="left", color=INK)
    ax.grid(True, alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(loc="best", frameon=False)

fig.savefig(OUTPUT_DIR / "inertia_estimates.png", dpi=200, bbox_inches="tight")
plt.show()

## Error summary

DEMO2's two error reports: errors are `estimate − ground truth` per snapshot; maximum and minimum
are signed, MAE and RMSE are not.

In [ ]:
def error_report(kind: str) -> dict:
    errors = (results[f"{kind} (MVA·s)"] - results["ground truth (MVA·s)"]).to_numpy()
    return {
        "snapshots": int(errors.size),
        "maximum error": float(np.max(errors)),
        "minimum error": float(np.min(errors)),
        "MAE": float(np.mean(np.abs(errors))),
        "RMSE": float(np.sqrt(np.mean(np.square(errors)))),
    }


if HAS_TRUTH:
    # One row per estimate, built from records so "snapshots" stays an integer column.
    reports = pd.DataFrame(
        [{"estimate": kind, **error_report(kind)} for kind in ("mean", "median")]
    ).set_index("estimate")
    display(reports.style.format(precision=1, thousands=","))
else:
    print("No ground truth, so no error summary.")

## Regional signal

These are the two series the Kalman filter regresses: power deviation against the rate of change of
frequency deviation, both measured from the first sample of each window. With several snapshots,
each gets its own panel and y-scale, plotted against time within the snapshot.

In [ ]:
labels = (
    [f"Hour {int(h)}" for h in results["hour"]] if results["hour"].notna().all()
    else [f"Window {w}" for w in results["window"]]
)
SIGNALS = [
    ("frequency_dev_pu", 60e3, "Frequency deviation (mHz)", "inertia-weighted frequency deviation"),
    ("power_dev_mw", 1.0, "Power deviation (MW)", "generation plus tie-line power deviation"),
]
n_windows = regional["window"].nunique()

if n_windows == 1:
    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, constrained_layout=True)
    t = regional["seconds"] - regional["seconds"].iloc[0]
    for axis, (column, scale, ylabel, title) in zip(axes, SIGNALS):
        axis.plot(t, regional[column] * scale, color=BLUE, linewidth=1.2)
        axis.set_ylabel(ylabel)
        axis.set_title(f"{REGION} {title}", loc="left", color=INK)
        axis.grid(True, alpha=0.25)
        axis.spines[["top", "right"]].set_visible(False)
    axes[-1].set_xlabel("Time (s)")
    fig.savefig(OUTPUT_DIR / "regional_signal.png", dpi=200, bbox_inches="tight")
    plt.show()
else:
    ncols = 6
    nrows = -(-n_windows // ncols)
    for column, scale, ylabel, title in SIGNALS:
        fig, axes = plt.subplots(nrows, ncols, figsize=(15, 2.1 * nrows), sharex=True,
                                 constrained_layout=True, squeeze=False)
        for axis in axes.flat[n_windows:]:
            axis.set_visible(False)
        for axis, (w, part) in zip(axes.flat, regional.groupby("window")):
            t = part["seconds"] - part["seconds"].iloc[0]
            axis.plot(t, part[column] * scale, color=BLUE, linewidth=0.9)
            axis.set_title(labels[w], loc="left", fontsize=9, color=INK)
            axis.tick_params(labelsize=7)
            axis.grid(True, alpha=0.25)
            axis.spines[["top", "right"]].set_visible(False)
        fig.supxlabel("Time within snapshot (s)")
        fig.supylabel(ylabel)
        fig.suptitle(f"{REGION} {title}, per snapshot (each panel has its own y-scale)",
                     x=0.01, ha="left", color=INK)
        fig.savefig(OUTPUT_DIR / f"regional_{column}.png", dpi=150, bbox_inches="tight")
        plt.show()

## Reading the result

- Each estimate is a mean or median of the filter's per-sample estimates over the whole window,
  including the first second before the disturbance (KNOWN_ISSUES.md #2).
- The power model fits poorly (median R² from −0.14 to 0.71 across DEMO2's hours, #1); DEMO2's
  estimates sit 10–19% below ground truth for most hours and 48–61% below for hours 13–14.
- The frequency weights use every machine in the region definition, not only the units running in
  each hour (#5); the ground truth counts only running units.
- In the 24-hour scenario, generators that are offline in an hour are replayed at 0 MW (#15).